# P07 统计基础：描述性统计与假设检验


## 本节你能学到什么
- 描述性统计（均值、中位数、方差、偏度、峰度）
- 正态分布直觉理解
- t 检验（单样本、双样本）
- 相关系数（Pearson）
- p 值与显著性水平怎么看
- 这些概念在 OLS 回归结果表格里怎么体现


---
## 1. 描述性统计


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams['font.family'] = ['Microsoft YaHei', 'SimHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

# 模拟 300 家上市公司的 ROE
np.random.seed(42)
roe = np.random.normal(loc=0.09, scale=0.08, size=300)
roe = pd.Series(roe, name='ROE')

print('=== 描述性统计 ===')
print(f'样本量:   {len(roe)}')
print(f'均值:     {roe.mean():.4f}  <- 平均水平')
print(f'中位数:   {roe.median():.4f}  <- 50%分位数')
print(f'标准差:   {roe.std():.4f}  <- 离散程度')
print(f'方差:     {roe.var():.4f}  <- 标准差的平方')
print(f'偏度:     {roe.skew():.4f}  <- >0 右偏(上尾长), <0 左偏')
print(f'峰度:     {roe.kurt():.4f}  <- >0 尖峰厚尾, 正态分布=0')
print(f'最小值:   {roe.min():.4f}')
print(f'最大值:   {roe.max():.4f}')


In [ ]:
# pandas 一行搞定
roe.describe().round(4)


---
## 2. 正态分布直觉
正态分布（钟形曲线）是最重要的概率分布：
- **68-95-99.7 法则**：均值 ± 1σ 包含 68% 数据，± 2σ 包含 95%，± 3σ 包含 99.7%
- OLS 回归假设误差项服从正态分布


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# 左：ROE 直方图 + 正态曲线
axes[0].hist(roe, bins=30, density=True, alpha=0.7, color='steelblue', edgecolor='white')
x = np.linspace(roe.min(), roe.max(), 200)
axes[0].plot(x, stats.norm.pdf(x, roe.mean(), roe.std()), 'r-', lw=2, label='理论正态')
axes[0].axvline(roe.mean(), color='black', linestyle='--', label=f'均值={roe.mean():.3f}')
axes[0].set_title('ROE 分布')
axes[0].legend()

# 右：QQ 图（判断是否正态）
stats.probplot(roe, plot=axes[1])
axes[1].set_title('QQ图（接近对角线 = 近似正态）')

plt.tight_layout()
plt.show()


---
## 3. 假设检验基础
**核心逻辑**（以 t 检验为例）：
1. **原假设 H₀**：「没有效果」「两组没差异」
2. **备择假设 H₁**：「有效果」「两组有差异」
3. 计算 **t 统计量**
4. 得到 **p 值**：在 H₀ 为真的前提下，观察到当前结果（或更极端结果）的概率
5. 若 p < 0.05（显著性水平 α），则拒绝 H₀，认为结果统计显著


### p 值怎么看？
| p 值 | 含义 | 论文中写法 |
|------|------|------------|
| p < 0.01 | 非常显著 | \*\*\* |
| 0.01 ≤ p < 0.05 | 显著 | \*\* |
| 0.05 ≤ p < 0.10 | 弱显著 | \* |
| p ≥ 0.10 | 不显著 | （无星号）|

> ⚠️ **p < 0.05 不代表效应大，只代表效应在统计上可信**（样本量越大越容易显著）


---
## 4. 单样本 t 检验
**问题**：A 股制造业企业的平均 ROE 是否显著高于 0.08（8%）？


In [ ]:
mfg_roe = np.random.normal(0.10, 0.07, 150)  # 制造业 ROE
mu0 = 0.08  # 原假设均值

t_stat, p_val = stats.ttest_1samp(mfg_roe, popmean=mu0)

print(f'样本均值: {mfg_roe.mean():.4f}')
print(f't 统计量: {t_stat:.4f}')
print(f'p 值（双尾）: {p_val:.4f}')
print(f'p 值（单尾，均值 > 0.08）: {p_val/2:.4f}')
print()
if p_val < 0.05:
    print('结论：在5%显著性水平下，拒绝原假设，制造业平均ROE显著不等于8%')
else:
    print('结论：在5%显著性水平下，无法拒绝原假设')


---
## 5. 双样本 t 检验
**问题**：制造业和金融业的平均 ROE 是否有显著差异？


In [ ]:
fin_roe = np.random.normal(0.13, 0.06, 120)   # 金融业 ROE

t_stat, p_val = stats.ttest_ind(mfg_roe, fin_roe, equal_var=False)  # Welch t 检验

print(f'制造业均值: {mfg_roe.mean():.4f}')
print(f'金融业均值: {fin_roe.mean():.4f}')
print(f'均值差: {mfg_roe.mean() - fin_roe.mean():.4f}')
print(f't 统计量: {t_stat:.4f}')
print(f'p 值: {p_val:.4f}')

sig = '***' if p_val < 0.01 else ('**' if p_val < 0.05 else ('*' if p_val < 0.1 else 'n.s.'))
print(f'显著性: {sig}')


---
## 6. 相关系数


In [ ]:
# 生成三个变量
np.random.seed(1)
n = 200
lev  = np.random.uniform(0.2, 0.8, n)           # 资产负债率
size = np.random.lognormal(5, 1, n)             # 规模
roe2 = 0.15 - 0.2*lev + 0.01*np.log(size) + np.random.normal(0, 0.05, n)

df = pd.DataFrame({'ROE': roe2, '杠杆率': lev, '规模(log)': np.log(size)})

print('相关系数矩阵:')
corr = df.corr().round(4)
print(corr)
print()
r, p = stats.pearsonr(roe2, lev)
print(f'ROE 与 杠杆率 的相关系数: r = {r:.4f}, p = {p:.4f}')


---
## 7. 这些统计量在 OLS 回归中的体现
运行一个简单的 OLS 回归，对照回归结果表格理解各统计量：


In [ ]:
import statsmodels.api as sm

X = sm.add_constant(df[['杠杆率', '规模(log)']])
y = df['ROE']

model = sm.OLS(y, X).fit()
print(model.summary())


In [ ]:
# 对照表：回归结果里的统计量
print('=== 回归结果解读 ===')
print()
print('coef   = 回归系数（效应大小）')
print('std err = 系数的标准误差（越小越精确）')
print('t      = coef / std err（t统计量）')
print('P>|t|  = p值（< 0.05 则显著）')
print('[0.025, 0.975] = 95%置信区间')
print()
print('R-squared     = 解释力（0-1，越高越好，但不唯一标准）')
print('Adj. R-squared= 调整后R方（考虑变量数量的惩罚）')
print('F-statistic   = 整体显著性检验（p<0.05则整体模型显著）')
print('AIC / BIC     = 模型选择标准（越小越好）')


---
## 练习题
1. 生成两组数据：A 组均值 50，B 组均值 52，标准差均为 5，各 30 个观测值。
   用双样本 t 检验判断两组均值是否显著不同（p < 0.05）。
   然后将每组增大到 300 个，再检验一次，观察 p 值的变化。
2. 对本节生成的 `df`（ROE、杠杆率、规模），画出相关系数热力图。
   - 提示：`import seaborn as sns; sns.heatmap(df.corr(), annot=True)`
3. 在上面的 OLS 回归结果中，找出：① 哪个变量最显著？② R² 是多少？③ F 检验 p 值是多少？
